# 🛡️ Tank Agent Lab — skid steering

In this notebook you will control a small robot tank in a 10 m × 10 m arena and then build an **agent** that drives it to a goal on its own.

An **agent** is something that *acts in an environment*. It works in a loop:

```
   ┌──────────── environment (arena, tank, goal) ────────────┐
   │                                                          │
percepts                                                   actions
   │                                                          │
   └──────►  controller  (the "brain": decides what to do) ───┘
```

* **Percepts** – what the agent can sense (`get_percepts()`)
* **Actions / commands** – what the agent can do (`agent_move(cmd, duration)`)
* **Controller** – the function that maps percepts to actions. First *you* will be the controller, then you'll write one.

👉 **Run the next cell first** (it builds the simulator). Then work through the notebook top to bottom.

In [ ]:
# @title 🔧 Tank simulator — run this cell first (you don't need to read it) { display-mode: "form" }
import math, random, time, io, contextlib, numbers
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle, Circle
from matplotlib.transforms import Affine2D
from IPython.display import display
from IPython import get_ipython

# ---------------- World constants ----------------
ARENA        = 10.0   # arena is ARENA x ARENA metres
TREAD_SPEED  = 0.5    # m/s of a tread at full forward
TRACK_WIDTH  = 0.6    # m between the two treads
TANK_RADIUS  = 0.45   # m, used for wall collisions
GOAL_RADIUS  = 0.5    # m, "close enough" to the goal
MAX_DURATION = 10.0   # s, longest single agent_move
PHYS_DT      = 0.02   # s, physics integration step
FRAME_DT     = 0.1    # s of simulated time between animation frames

SETTINGS = {
    "animate": True,       # draw the tank while it moves
    "sim_speed": 1.0,      # 1.0 = real time, 2.0 = twice as fast, ...
    "motion_noise": 0.0,   # 0.0 = perfect treads; try 0.2 later!
}

# ---------------- The 9 commands ----------------
# Each command is (left_tread, right_tread), each one of
#   +1 = Forward, 0 = Neutral, -1 = Backward (reverse)
# Code letters: first letter = LEFT tread, second = RIGHT tread.
COMMANDS = {}
_LETTER = {1: "F", 0: "N", -1: "B"}
for _l in (-1, 0, 1):
    for _r in (-1, 0, 1):
        COMMANDS[_LETTER[_l] + _LETTER[_r]] = (_l, _r)
COMMAND_LIST = list(COMMANDS)          # index 0..8 -> code, e.g. 0 = 'BB', 8 = 'FF'


def parse_cmd(cmd):
    """Accepts 'FB', ('F','B'), (1,-1) or an index 0..8. Returns (left, right)."""
    if isinstance(cmd, str):
        code = cmd.replace(" ", "").replace(",", "").upper()
        if code in COMMANDS:
            return COMMANDS[code]
        if code.isdigit() and 0 <= int(code) <= 8:
            return COMMANDS[COMMAND_LIST[int(code)]]
    elif isinstance(cmd, numbers.Integral) and not isinstance(cmd, bool) and 0 <= cmd <= 8:
        return COMMANDS[COMMAND_LIST[int(cmd)]]
    elif isinstance(cmd, (tuple, list)) and len(cmd) == 2:
        vals = []
        for v in cmd:
            if isinstance(v, str) and v.upper() in ("F", "N", "B"):
                v = {"F": 1, "N": 0, "B": -1}[v.upper()]
            if v not in (-1, 0, 1):
                break
            vals.append(int(v))
        else:
            return tuple(vals)
    raise ValueError(f"Unknown command {cmd!r}. Use one of {COMMAND_LIST}, "
                     "a pair like (1, -1), or an index 0-8.")


def show_commands():
    """Print the 9 commands as a 3x3 grid."""
    print("            RIGHT tread")
    print("             F     N     B")
    for l_letter, l in (("F", 1), ("N", 0), ("B", -1)):
        row = "   ".join(f"{l_letter + r_letter:>3}" for r_letter in "FNB")
        lead = "LEFT tread " if l == 0 else "           "
        print(f"{lead}{l_letter}  {row}")
    print("\nF = forward, N = neutral (stopped), B = backward (reverse)")
    print("Index form: ", {i: c for i, c in enumerate(COMMAND_LIST)})


class OutOfTime(Exception):
    """Raised when an agent uses more simulated time than a test allows."""


world = {}


# ---------------- Drawing ----------------
_TREAD_COLOR = {1: "#2ca02c", 0: "#7f7f7f", -1: "#d62728"}   # green / grey / red

def _build_figure():
    fig, ax = plt.subplots(figsize=(6, 6), dpi=72)
    plt.close(fig)                       # we display it ourselves
    ax.set_xlim(0, ARENA); ax.set_ylim(0, ARENA); ax.set_aspect("equal")
    ax.set_xticks(range(0, int(ARENA) + 1)); ax.set_yticks(range(0, int(ARENA) + 1))
    ax.grid(True, color="#e5e5e5", zorder=0)
    ax.set_facecolor("#fbfaf5")
    for s in ax.spines.values():
        s.set_linewidth(3); s.set_color("#444")

    art = {"fig": fig, "ax": ax}
    art["goal"] = Circle((0, 0), GOAL_RADIUS, fc="#ffd54f", ec="#b28704", lw=2, zorder=1)
    ax.add_patch(art["goal"])
    art["goal_label"] = ax.text(0, 0, "GOAL", ha="center", va="center",
                                fontsize=8, weight="bold", zorder=2)
    (art["trail"],) = ax.plot([], [], ":", color="#1f77b4", lw=1.5, zorder=2)

    # Tank parts drawn in tank coordinates: +x = forward, +y = the tank's left side.
    L, W, T = 0.8, 0.6, 0.16      # body length, overall width, tread width
    art["body"]  = Rectangle((-L/2 + 0.05, -W/2 + T), L - 0.1, W - 2*T,
                             fc="#556b2f", ec="black", lw=1, zorder=4)
    art["left"]  = Rectangle((-L/2,  W/2 - T), L, T, ec="black", lw=1, zorder=4)
    art["right"] = Rectangle((-L/2, -W/2),     L, T, ec="black", lw=1, zorder=4)
    art["turret"] = Circle((-0.05, 0), 0.15, fc="#6b8e23", ec="black", lw=1, zorder=6)
    art["barrel"] = Rectangle((0.05, -0.03), 0.42, 0.06, fc="#333", ec="black", lw=0.5, zorder=5)
    for k in ("body", "left", "right", "turret", "barrel"):
        ax.add_patch(art[k])
    art["status"] = ax.text(0.01, 1.01, "", transform=ax.transAxes, fontsize=9,
                            family="monospace", va="bottom")
    return art


def _render():
    a, w = world["art"], world
    x, y, th = w["x"], w["y"], w["theta"]
    tf = Affine2D().rotate(th).translate(x, y) + a["ax"].transData
    for k in ("body", "left", "right", "turret", "barrel"):
        a[k].set_transform(tf)
    l, r = w["treads"]
    a["left"].set_facecolor(_TREAD_COLOR[l])
    a["right"].set_facecolor(_TREAD_COLOR[r])

    gx, gy = w["goal"]
    a["goal"].center = (gx, gy)
    a["goal"].set_facecolor("#8bc34a" if _at_goal() else "#ffd54f")
    a["goal_label"].set_position((gx, gy))
    xs, ys = zip(*w["trail"])
    a["trail"].set_data(xs, ys)

    code = _LETTER[l] + _LETTER[r]
    msg = f"t={w['t']:6.1f}s  cmd={code}  x={x:4.1f} y={y:4.1f} hdg={math.degrees(th) % 360:5.1f}°"
    if w["bumped"]: msg += "  BUMP!"
    if _at_goal():  msg += "  GOAL!"
    a["status"].set_text(msg)
    _show()


def _show():
    """Draw into the output of the cell that is currently running."""
    if world.get("handle") is None:
        world["handle"] = display(world["art"]["fig"], display_id=True)
    else:
        world["handle"].update(world["art"]["fig"])


def _new_cell(*args):
    """Called by IPython before every cell runs: start a fresh picture in that cell."""
    world["handle"] = None

_ip = get_ipython()
if _ip is not None:
    _cbs = _ip.events.callbacks["pre_run_cell"]
    for _cb in [c for c in _cbs if getattr(c, "__name__", "") == "_new_cell"]:
        _ip.events.unregister("pre_run_cell", _cb)     # avoid duplicates if re-run
    _ip.events.register("pre_run_cell", _new_cell)


# ---------------- Physics ----------------
def _at_goal():
    gx, gy = world["goal"]
    return math.hypot(world["x"] - gx, world["y"] - gy) <= GOAL_RADIUS


def _step_physics(left, right, dt, slip=(1.0, 1.0)):
    vl = left * TREAD_SPEED * slip[0]
    vr = right * TREAD_SPEED * slip[1]
    v = (vl + vr) / 2                   # forward speed of the tank's centre
    omega = (vr - vl) / TRACK_WIDTH     # turn rate (rad/s), + = counter-clockwise
    w = world
    w["theta"] = (w["theta"] + omega * dt + math.pi) % (2 * math.pi) - math.pi
    nx = w["x"] + v * math.cos(w["theta"]) * dt
    ny = w["y"] + v * math.sin(w["theta"]) * dt
    lo, hi = TANK_RADIUS, ARENA - TANK_RADIUS
    if not (lo <= nx <= hi and lo <= ny <= hi):
        w["bumped"] = True
    w["x"] = min(max(nx, lo), hi)
    w["y"] = min(max(ny, lo), hi)
    w["t"] += dt
    px, py = w["trail"][-1]
    if math.hypot(w["x"] - px, w["y"] - py) > 0.05:
        w["trail"].append((w["x"], w["y"]))


def _front_range():
    """Distance from the tank's centre to the wall straight ahead."""
    x, y, th = world["x"], world["y"], world["theta"]
    c, s = math.cos(th), math.sin(th)
    ds = []
    if c > 1e-9:  ds.append((ARENA - x) / c)
    if c < -1e-9: ds.append(-x / c)
    if s > 1e-9:  ds.append((ARENA - y) / s)
    if s < -1e-9: ds.append(-y / s)
    return min(ds)


# ================= PUBLIC FUNCTIONS =================
def reset_world(goal=None, start=(2.0, 2.0, 0.0), seed=None):
    """Start a new episode.
    start = (x, y, heading_in_degrees)   heading 0 = east/right, 90 = north/up
    goal  = (x, y), or None for a random goal (seed=... makes it repeatable)."""
    rng = random.Random(seed)
    sx, sy, sh = start
    lo, hi = TANK_RADIUS, ARENA - TANK_RADIUS
    if not (lo <= sx <= hi and lo <= sy <= hi):
        raise ValueError(f"start x and y must be between {lo} and {hi}")
    if goal is not None and not all(0 <= g <= ARENA for g in goal):
        raise ValueError(f"goal x and y must be between 0 and {ARENA}")
    if goal is None:
        while True:
            goal = (rng.uniform(1, ARENA - 1), rng.uniform(1, ARENA - 1))
            if math.hypot(goal[0] - sx, goal[1] - sy) > 3:
                break
    if "art" not in world:
        world["art"] = _build_figure()
    world.update(x=float(sx), y=float(sy), theta=math.radians(sh), goal=tuple(goal),
                 t=0.0, treads=(0, 0), bumped=False, trail=[(sx, sy)],
                 time_limit=world.get("time_limit"))
    if SETTINGS["animate"]:
        _render()


def get_percepts():
    """What the agent can sense. Angles are in degrees.
    goal_bearing: where the goal is relative to where the tank points,
                  from -180 to 180; positive = goal is to the LEFT."""
    gx, gy = world["goal"]
    dx, dy = gx - world["x"], gy - world["y"]
    bearing = math.degrees(math.atan2(dy, dx) - world["theta"])
    bearing = (bearing + 180) % 360 - 180
    return {
        "x": round(world["x"], 3),
        "y": round(world["y"], 3),
        "heading": round(math.degrees(world["theta"]) % 360, 2),   # 0 = east, 90 = north
        "goal_distance": round(math.hypot(dx, dy), 3),
        "goal_bearing": round(bearing, 2),
        "front_range": round(_front_range(), 3),
        "bumped": world["bumped"],
        "at_goal": _at_goal(),
        "time": round(world["t"], 2),
    }


def agent_move(cmd, duration):
    """Run the treads with command `cmd` for `duration` seconds, then stop.
    cmd: one of the 9 codes 'FF','FN','FB','NF','NN','NB','BF','BN','BB'
         (first letter = LEFT tread, second = RIGHT tread),
         or a pair like (1, -1), or an index 0-8.
    Returns the percepts after the move."""
    if "art" not in world:
        reset_world()
    left, right = parse_cmd(cmd)
    duration = float(duration)
    if not (0 < duration <= MAX_DURATION):
        raise ValueError(f"duration must be between 0 and {MAX_DURATION} seconds")

    world["treads"] = (left, right)
    world["bumped"] = False
    animate = SETTINGS["animate"]
    noise = SETTINGS["motion_noise"]          # each tread slips a random amount this move
    slip = (max(0.0, 1 + random.gauss(0, noise)), max(0.0, 1 + random.gauss(0, noise)))
    elapsed, next_frame = 0.0, FRAME_DT
    wall_start = time.time()
    while elapsed < duration - 1e-9:
        dt = min(PHYS_DT, duration - elapsed)
        _step_physics(left, right, dt, slip)
        elapsed += dt
        limit = world.get("time_limit")
        if limit is not None and world["t"] > limit:
            raise OutOfTime(f"agent used more than {limit} s of simulated time")
        if animate and elapsed >= next_frame - 1e-9:
            _render()
            next_frame += FRAME_DT
            lag = wall_start + elapsed / SETTINGS["sim_speed"] - time.time()
            if lag > 0:
                time.sleep(lag)
    world["treads"] = (0, 0)
    if animate:
        _render()
    return get_percepts()


def human_controller():
    """YOU are the controller: type a command and a duration, e.g.  FF 2"""
    reset_world()
    print("Type:  <cmd> <seconds>   e.g.  FF 2   or   BF 0.5   (seconds defaults to 1)")
    print("Other: 'p' = show percepts, 'reset' = new random goal, 'help', 'q' = quit\n")
    show_commands()
    while True:
        try:
            line = input("\ncmd duration> ").strip()
        except (EOFError, KeyboardInterrupt):
            break
        if not line:
            continue
        word = line.lower()
        if word in ("q", "quit", "exit"):
            break
        if word in ("h", "help", "?"):
            show_commands(); continue
        if word in ("p", "percepts"):
            print(get_percepts()); continue
        if word == "reset":
            reset_world(); continue
        parts = line.split()
        if len(parts) > 2:
            print("⚠️  Type the command as ONE word, then the seconds, e.g.  FB 1.5"); continue
        try:
            duration = float(parts[1]) if len(parts) > 1 else 1.0
        except ValueError:
            print(f"⚠️  {parts[1]!r} is not a number of seconds"); continue
        try:
            p = agent_move(parts[0], duration)
        except ValueError as e:
            print("⚠️ ", e); continue
        print(f"goal_distance={p['goal_distance']:.2f} m  goal_bearing={p['goal_bearing']:+.1f}°"
              + ("  BUMPED the wall!" if p["bumped"] else ""))
        if p["at_goal"]:
            print(f"🎉 You reached the goal in {p['time']:.1f} s of driving!")
    print("Human controller finished.")


def test_agent(controller, trials=10, time_limit=120, noise=None, goal=None, start=None,
               verbose=False):
    """Run `controller()` `trials` times without animation and report results.
    By default every trial has a different random start (position + heading) and goal;
    pass goal=(x, y) and/or start=(x, y, heading) to fix them.
    Anything the controller prints is hidden unless verbose=True."""
    old = dict(SETTINGS)
    SETTINGS["animate"] = False
    if noise is not None:
        SETTINGS["motion_noise"] = noise
    results = []
    try:
        for i in range(trials):
            world["time_limit"] = time_limit
            r = random.Random(i)
            st = start if start is not None else (r.uniform(1, ARENA - 1), r.uniform(1, ARENA - 1),
                                                  r.uniform(0, 360))
            reset_world(goal=goal, start=st, seed=1000 + i)
            random.seed(2000 + i)          # makes noisy runs repeatable
            status = ""
            out = None if verbose else io.StringIO()
            try:
                with contextlib.redirect_stdout(out) if out else contextlib.nullcontext():
                    controller()
            except OutOfTime:
                status = "(out of time)"
            p = get_percepts()
            results.append(p["at_goal"])
            print(f"trial {i:2d}: start=({st[0]:4.1f},{st[1]:4.1f}) goal=({world['goal'][0]:4.1f},{world['goal'][1]:4.1f})  "
                  f"{'REACHED' if p['at_goal'] else 'missed '}  time={p['time']:6.1f}s  "
                  f"final distance={p['goal_distance']:5.2f} m  {status}")
    finally:
        world["time_limit"] = None
        SETTINGS.update(old)
    print(f"Success rate: {sum(results)}/{trials}")


reset_world(seed=0)
print("Simulator ready.")

## 1. The tank's body: skid steering

The tank has no steering wheel. It has two **treads** (tracks), and each tread can be in one of three states:

| symbol | meaning |
|---|---|
| **F** | forward |
| **N** | neutral (stopped) |
| **B** | backward (reverse) |

2 treads × 3 states = **9 commands**. A command is written as two letters: **first letter = LEFT tread, second letter = RIGHT tread**. So `FB` means *left tread forward, right tread backward*.

**Left and right are from the tank's point of view** — imagine sitting in it, facing the way the barrel points. When the tank faces down the screen, its left tread is on *your* right!

The tank turns by running its treads at different speeds — this is called **skid steering** (bulldozers, tanks, and many robot vacuums work this way).

In the picture, each tread is coloured by its state: 🟩 forward, ⬜ neutral (grey), 🟥 backward. The dotted blue line is the tank's trail.

In [ ]:
show_commands()

### ✏️ Exercise 1 — predict before you test

For each command, predict what the tank will do *before* you run it: drive straight? spin in place? pivot around one tread? arc forward or backward, and to which side?

| cmd | your prediction | what actually happened |
|---|---|---|
| FF | | |
| BB | | |
| FB | | |
| BF | | |
| FN | | |
| NF | | |
| BN | | |
| NB | | |
| NN | | |

Now test them. `agent_move(cmd, duration)` runs one command for `duration` seconds (simulated time, max 10) and then stops the treads. `reset_world(...)` puts the tank back at the start. Edit and re-run this cell:

In [ ]:
reset_world(seed=0)
agent_move("FF", 2)     # left tread Forward, right tread Forward, for 2 seconds
agent_move("FN", 2)     # try other commands here!

## 2. You are the controller

`human_controller()` puts **you** in the agent loop. You look at the picture (your percepts), decide, and type a command and a duration, e.g.

```
FF 3
BF 0.5
```

Also: `p` prints the percepts, `reset` starts over with a new goal, `q` quits. Try to reach the yellow goal circle (it turns green when you're there).

> ⚠️ Type `q` when you're done. Until you quit, this cell keeps running and **no other cell can run**.

In [ ]:
human_controller()

### 🤔 Think about it
1. What information were **you** using to decide? Did you use the picture, the numbers printed after each move, or both?
2. Did you plan the whole route in advance, or look → move a bit → look again? Which worked better?

## 3. Percepts: what a program can sense

A program can't *see* the picture. It only gets the numbers from `get_percepts()`:

| key | meaning |
|---|---|
| `x`, `y` | position in metres (0–10) |
| `heading` | direction the tank points, degrees: 0 = east (right), 90 = north (up) |
| `goal_distance` | metres from tank centre to goal centre |
| `goal_bearing` | where the goal is **relative to the barrel**, −180…180°. **Positive = goal is to the LEFT**, negative = to the right, 0 = dead ahead |
| `front_range` | metres from the tank's centre to the wall straight ahead (the tank touches a wall head-on at about 0.45) |
| `bumped` | `True` if the last move hit a wall |
| `at_goal` | `True` if within 0.5 m of the goal |
| `time` | simulated seconds used so far |

Useful constants: `TREAD_SPEED` = 0.5 m/s (so `FF` drives 0.5 m per second) and `TRACK_WIDTH` = 0.6 m.
`agent_move` also **returns** the percepts after moving.

In [ ]:
reset_world(seed=0)
get_percepts()

## 4. Two example agents (that are not very good)

**A blind (open-loop) agent** follows a fixed plan and never looks at its percepts. This one works — but only for one particular start and goal.

In [ ]:
def blind_agent():
    # Plan worked out by hand for: start (2, 2) facing east, goal at (4, 4)
    agent_move("FF", 4)        # 4 s x 0.5 m/s = 2 m east
    agent_move("BF", 0.94)     # spin left ~90 degrees
    agent_move("FF", 4)        # 2 m north

reset_world(goal=(4, 4), start=(2, 2, 0))
blind_agent()
print(get_percepts())

**A random agent** senses only to check whether it's done — its choices ignore the percepts.

In [ ]:
def random_agent(max_steps=15):
    for step in range(max_steps):
        if get_percepts()["at_goal"]:
            print("Lucky!")
            return True
        agent_move(random.choice(COMMAND_LIST), 0.5)
    return False

reset_world(seed=0)
random_agent()

## 5. ✏️ Your turn: write `agent_controller`

Write a controller that reaches the goal **from any start and any goal location**. Keep the name `agent_controller` and the `max_steps` loop (so a stuck agent eventually stops). The template below already has the **sense → decide → act** loop; you only need to fill in the *decide* part.

Hints:
* Start with the simplest idea that could work. What should the tank do if `goal_bearing` is large and positive? Large and negative? Close to zero?
* Short actions (≤ 1 s) let your agent re-sense and correct its mistakes.
* Spinning in place with `BF`/`FB` turns about 95°/s. How long should you spin to turn 45°?

In [ ]:
def agent_controller(max_steps=30):
    """YOUR AGENT. Repeatedly: sense -> decide -> act."""
    for step in range(max_steps):
        p = get_percepts()                      # 1. SENSE

        if p["at_goal"]:
            print(f"Goal reached at t = {p['time']} s")
            return True

        # 2. DECIDE  (replace this with your own rules!)
        cmd = "NN"
        duration = 0.5

        agent_move(cmd, duration)               # 3. ACT
    print("Ran out of steps.")
    return False

Watch your agent drive:

In [ ]:
reset_world(seed=0)
agent_controller()

Try a few other goals: change the seed, or set them yourself, e.g. `reset_world(goal=(8, 3), start=(5, 5, 180))` puts the tank at (5, 5) facing 180° (west) with the goal at (8, 3).

## 6. Test your agent

A single success could be luck. `test_agent` runs your controller 10 times, each with a different random start position, heading and goal, **without animation** (fast), and reports the success rate. Each trial is limited to 120 s of simulated time.

(Anything your controller prints is hidden during the test; use `test_agent(agent_controller, verbose=True)` to see it.)

In [ ]:
test_agent(agent_controller, trials=10)

## 7. When the world isn't perfect: noisy treads

Real treads slip. Setting `noise=0.2` makes each tread's speed randomly off by about 20% on every move.

Let's give the blind agent its *own* start and goal, first with perfect treads and then with slipping treads:

In [ ]:
print("--- blind agent, its own goal, perfect treads ---")
test_agent(blind_agent, trials=5, goal=(4, 4), start=(2, 2, 0), noise=0.0)
print("\n--- blind agent, its own goal, noisy treads ---")
test_agent(blind_agent, trials=5, goal=(4, 4), start=(2, 2, 0), noise=0.2)
print("\n--- blind agent, random goals ---")
test_agent(blind_agent, trials=5)
print("\n--- YOUR agent, noisy treads ---")
test_agent(agent_controller, trials=10, noise=0.2)

### 🤔 Questions
1. The blind agent's plan is perfect for its goal — so why does it fail when the treads slip? Why does it fail on random goals?  What's the difference between an **open-loop** and a **closed-loop** (feedback) controller?
2. Does your agent still work with noise? If not, what would make it more robust?
3. Is your agent **reactive** (decides only from the current percepts) or does it keep **memory / a plan**? Would memory help here?
4. **Challenge:** make your agent faster. Can it drive and turn at the same time (arcs with `FN`/`NF`) instead of stopping to spin? Compare the average time.
5. **Challenge:** what would change if the tank could only sense `front_range` and `bumped` — no goal information? What task could it still do?

💡 Tip: set `SETTINGS["sim_speed"] = 3` to speed up the animation, or `SETTINGS["animate"] = False` to turn it off.